In [9]:
# ============================================================
# BLOQUE 0. INSTALACIÓN Y CARGA DE LIBRERÍAS
# Proyecto: Cultura ciudadana - Encuesta Calidad de Vida USCO
# Uso: Google Colab
# ============================================================

# Instalar paquetes necesarios
!pip install -q openpyxl unidecode adjustText

# Librerías base
import os
import re
import zipfile
import warnings
from datetime import datetime

# Manejo de datos
import numpy as np
import pandas as pd

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Estadística
from scipy.stats import chi2_contingency
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

# Utilidades Colab
from google.colab import files

# Normalización de nombres
from unidecode import unidecode

# Configuración general
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 120)

# Estilo visual profesional
sns.set_theme(
    style="whitegrid",
    context="notebook",
    font_scale=1.05
)

plt.rcParams["figure.figsize"] = (11, 6)
plt.rcParams["axes.titlesize"] = 15
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["font.family"] = "DejaVu Sans"

print("✅ Librerías cargadas correctamente.")

✅ Librerías cargadas correctamente.


In [10]:
# ============================================================
# BLOQUE 0. INSTALACIÓN Y CARGA DE LIBRERÍAS
# Proyecto: Cultura ciudadana - Encuesta Calidad de Vida USCO
# Uso: Google Colab
# ============================================================

# Instalar paquetes necesarios
!pip install -q openpyxl unidecode adjustText

# Librerías base
import os
import re
import zipfile
import warnings
from datetime import datetime

# Manejo de datos
import numpy as np
import pandas as pd

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Estadística
from scipy.stats import chi2_contingency
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

# Utilidades Colab
from google.colab import files

# Normalización de nombres
from unidecode import unidecode

# Configuración general
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 120)

# Estilo visual profesional
sns.set_theme(
    style="whitegrid",
    context="notebook",
    font_scale=1.05
)

plt.rcParams["figure.figsize"] = (11, 6)
plt.rcParams["axes.titlesize"] = 15
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["font.family"] = "DejaVu Sans"

print("✅ Librerías cargadas correctamente.")

✅ Librerías cargadas correctamente.


In [11]:
# ============================================================
# BLOQUE 2. CREACIÓN DE CARPETAS DE SALIDA
# ============================================================

CARPETA_SALIDA = "resultados_cultura_ciudadana"
CARPETA_GRAFICOS = os.path.join(CARPETA_SALIDA, "graficos")
CARPETA_TABLAS = os.path.join(CARPETA_SALIDA, "tablas")
CARPETA_INFORME = os.path.join(CARPETA_SALIDA, "informe")

for carpeta in [CARPETA_SALIDA, CARPETA_GRAFICOS, CARPETA_TABLAS, CARPETA_INFORME]:
    os.makedirs(carpeta, exist_ok=True)

print("✅ Carpetas creadas:")
print(CARPETA_SALIDA)
print(CARPETA_GRAFICOS)
print(CARPETA_TABLAS)
print(CARPETA_INFORME)

✅ Carpetas creadas:
resultados_cultura_ciudadana
resultados_cultura_ciudadana/graficos
resultados_cultura_ciudadana/tablas
resultados_cultura_ciudadana/informe


In [12]:
# ============================================================
# BLOQUE 3. NORMALIZACIÓN Y CONCILIACIÓN DE NOMBRES
# ============================================================

def normalizar_texto(x):
    """
    Convierte texto a una versión simple:
    - minúsculas
    - sin tildes
    - sin espacios extremos
    - espacios y caracteres raros convertidos a guion bajo
    """
    x = str(x).strip().lower()
    x = unidecode(x)
    x = re.sub(r"\s+", "_", x)
    x = re.sub(r"[^\w_]", "_", x)
    x = re.sub(r"_+", "_", x)
    x = x.strip("_")
    return x

# Crear mapa entre columnas originales y normalizadas
columnas_originales = list(df.columns)
mapa_normalizado = {normalizar_texto(col): col for col in columnas_originales}

# Mostrar columnas originales para revisar
print("📌 Primeras columnas originales de la base:")
print(columnas_originales[:30])

print("\n✅ Mapa de columnas normalizadas creado.")

📌 Primeras columnas originales de la base:
['municipio', 'an', 'sexo', 'edad', 'nivels', 'percepción', 'orgullo', 'satisfacción', 'futuro', 'economía', 'plaboral', 'ftrabajo', 'pobre', 'spobre', 'npobre', 'comida', 'infancia', 'secundaria', 'superior', 'pública', 'privada', 'salud', 'ssalud', 'mseguridad', 'bseguridad', 'prseguridad', 'delito', 'denuncia', 'sanción', 'barrio']

✅ Mapa de columnas normalizadas creado.


In [13]:
# ============================================================
# BLOQUE 4. DEFINICIÓN DE VARIABLES DEL BLOQUE CULTURA CIUDADANA
# ============================================================

# Variables principales del bloque Cultura Ciudadana
variables_cultura_esperadas = [
    "honestidad",
    "convivencia",
    "bienes",
    "transito",
    "ambiental",
    "migrantes",
    "pensamiento",
    "impuestos",
    "arrojar",
    "conexion",
    "orinar",
    "indebido",
    "construccion",
    "volumen",
    "compotro"
]

# Variables transversales obligatorias
variables_transversales_esperadas = [
    "municipio",
    "sexo",
    "nivels",
    "edad",
    "nedu",
    "an"
]

# Diccionario de preguntas / significado
diccionario_cultura = {
    "honestidad": "Cumplimiento de normas: honestidad/legalidad en conexión a servicios públicos",
    "convivencia": "Cumplimiento de normas básicas de convivencia",
    "bienes": "Cuidado y respeto de espacios y bienes públicos",
    "transito": "Respeto a normas básicas de tránsito",
    "ambiental": "Respeto a normas ambientales",
    "migrantes": "Cumplimiento de normas / comportamiento frente a migrantes o refugiados",
    "pensamiento": "Cumplimiento de normas / respeto frente a quienes piensan políticamente diferente",
    "impuestos": "Cree que puede ser castigado por no pagar impuestos",
    "arrojar": "Cree que puede ser castigado por arrojar basura o escombros al espacio público",
    "conexion": "Cree que puede ser castigado por conectarse ilegalmente a servicios públicos",
    "orinar": "Cree que puede ser castigado por orinar en el espacio público",
    "indebido": "Cree que puede ser castigado por ocupar indebidamente el espacio público",
    "construccion": "Cree que puede ser castigado por violar una norma de construcción",
    "volumen": "Cree que puede ser castigado por poner música a alto volumen",
    "compotro": "Cree que puede ser castigado por otro comportamiento"
}

# Nombres cortos para gráficos
nombres_cortos = {
    "honestidad": "Conexión legal a servicios",
    "convivencia": "Convivencia",
    "bienes": "Bienes públicos",
    "transito": "Normas de tránsito",
    "ambiental": "Normas ambientales",
    "migrantes": "Migrantes/refugiados",
    "pensamiento": "Diferencia política",
    "impuestos": "Castigo por impuestos",
    "arrojar": "Castigo por basura",
    "conexion": "Castigo por conexión ilegal",
    "orinar": "Castigo por orinar",
    "indebido": "Castigo por ocupación indebida",
    "construccion": "Castigo por construcción",
    "volumen": "Castigo por alto volumen",
    "compotro": "Castigo por otro comportamiento"
}

# Clasificación temática interna
grupo_variable = {
    "honestidad": "Cumplimiento de normas",
    "convivencia": "Cumplimiento de normas",
    "bienes": "Cumplimiento de normas",
    "transito": "Cumplimiento de normas",
    "ambiental": "Cumplimiento de normas",
    "migrantes": "Cumplimiento de normas",
    "pensamiento": "Cumplimiento de normas",
    "impuestos": "Percepción de sanción",
    "arrojar": "Percepción de sanción",
    "conexion": "Percepción de sanción",
    "orinar": "Percepción de sanción",
    "indebido": "Percepción de sanción",
    "construccion": "Percepción de sanción",
    "volumen": "Percepción de sanción",
    "compotro": "Percepción de sanción"
}

print("✅ Variables del bloque definidas.")

✅ Variables del bloque definidas.


In [14]:
# ============================================================
# BLOQUE 5. CONCILIACIÓN DE VARIABLES CON LA BASE REAL
# ============================================================

def encontrar_columna(nombre_esperado, mapa_cols):
    """
    Busca una variable esperada dentro de las columnas reales de la base.
    Permite diferencias de tilde, mayúsculas, espacios, etc.
    """
    nombre_norm = normalizar_texto(nombre_esperado)

    # Búsqueda directa normalizada
    if nombre_norm in mapa_cols:
        return mapa_cols[nombre_norm]

    # Búsqueda flexible por contenido exacto parcial
    candidatos = []
    for col_norm, col_original in mapa_cols.items():
        if nombre_norm == col_norm:
            candidatos.append(col_original)
        elif nombre_norm in col_norm:
            candidatos.append(col_original)

    if len(candidatos) > 0:
        return candidatos[0]

    return None

# Conciliar variables cultura
mapa_cultura = {}
variables_no_encontradas = []

for var in variables_cultura_esperadas:
    col_real = encontrar_columna(var, mapa_normalizado)
    if col_real is not None:
        mapa_cultura[var] = col_real
    else:
        variables_no_encontradas.append(var)

# Conciliar transversales
mapa_transversales = {}
transversales_no_encontradas = []

for var in variables_transversales_esperadas:
    col_real = encontrar_columna(var, mapa_normalizado)
    if col_real is not None:
        mapa_transversales[var] = col_real
    else:
        transversales_no_encontradas.append(var)

print("✅ Variables de cultura encontradas:")
for k, v in mapa_cultura.items():
    print(f"{k:15s} -> {v}")

print("\n⚠️ Variables de cultura NO encontradas:")
print(variables_no_encontradas)

print("\n✅ Variables transversales encontradas:")
for k, v in mapa_transversales.items():
    print(f"{k:15s} -> {v}")

print("\n⚠️ Variables transversales NO encontradas:")
print(transversales_no_encontradas)

# Crear tabla de conciliación para el informe
tabla_conciliacion = pd.DataFrame({
    "variable_esperada": list(mapa_cultura.keys()) + list(mapa_transversales.keys()),
    "variable_real_en_base": list(mapa_cultura.values()) + list(mapa_transversales.values())
})

ruta_conciliacion = os.path.join(CARPETA_TABLAS, "tabla_conciliacion_variables.xlsx")
tabla_conciliacion.to_excel(ruta_conciliacion, index=False)

display(tabla_conciliacion)

print(f"✅ Tabla de conciliación exportada: {ruta_conciliacion}")

✅ Variables de cultura encontradas:
honestidad      -> honestidad
convivencia     -> convivencia
bienes          -> bienes
transito        -> tránsito
ambiental       -> ambiental
migrantes       -> migrantes
pensamiento     -> pensamiento
impuestos       -> impuestos
arrojar         -> arrojar
conexion        -> conexión
orinar          -> orinar
indebido        -> indebido
construccion    -> construcción
volumen         -> volumen
compotro        -> compotro

⚠️ Variables de cultura NO encontradas:
[]

✅ Variables transversales encontradas:
municipio       -> municipio
sexo            -> sexo
nivels          -> nivels
edad            -> edad
nedu            -> nedu
an              -> an

⚠️ Variables transversales NO encontradas:
[]


,variable_esperada,variable_real_en_base
0,honestidad,honestidad
1,convivencia,convivencia
2,bienes,bienes
3,transito,tránsito
4,ambiental,ambiental
5,migrantes,migrantes
6,pensamiento,pensamiento
7,impuestos,impuestos
8,arrojar,arrojar
9,conexion,conexión


✅ Tabla de conciliación exportada: resultados_cultura_ciudadana/tablas/tabla_conciliacion_variables.xlsx


In [15]:
# ============================================================
# BLOQUE 6. CREACIÓN DE BASE LIMPIA DE TRABAJO
# ============================================================

# Columnas reales necesarias
columnas_cultura_reales = list(mapa_cultura.values())
columnas_transversales_reales = list(mapa_transversales.values())

columnas_usar = columnas_transversales_reales + columnas_cultura_reales
columnas_usar = list(dict.fromkeys(columnas_usar))  # eliminar duplicados manteniendo orden

df_cc = df[columnas_usar].copy()

# Renombrar columnas reales a nombres estandarizados
renombrar = {}
for var_estandar, col_real in mapa_cultura.items():
    renombrar[col_real] = var_estandar

for var_estandar, col_real in mapa_transversales.items():
    renombrar[col_real] = var_estandar

df_cc = df_cc.rename(columns=renombrar)

print("✅ Base de trabajo creada.")
print(f"Dimensiones base cultura ciudadana: {df_cc.shape[0]:,} filas x {df_cc.shape[1]:,} columnas")

display(df_cc.head())

✅ Base de trabajo creada.
Dimensiones base cultura ciudadana: 6,745 filas x 21 columnas


,municipio,sexo,nivels,edad,nedu,an,honestidad,convivencia,bienes,transito,ambiental,migrantes,pensamiento,impuestos,arrojar,conexion,orinar,indebido,construccion,volumen,compotro
0,1,0,1,31,NaN,2021,1,1,1,0,1,NaN,NaN,1,1,1,1,1,1,1,NaN
1,1,1,0,21,NaN,2021,0,0,0,0,0,NaN,NaN,0,1,1,1,0,0,1,NaN
2,1,1,1,27,NaN,2021,0,1,0,0,0,NaN,NaN,1,1,1,1,1,1,1,NaN
3,1,1,1,19,NaN,2021,0,0,0,0,0,NaN,NaN,1,1,1,1,1,1,0,NaN
4,1,1,0,22,NaN,2021,1,1,0,0,1,NaN,NaN,0,0,0,0,0,0,0,NaN


In [16]:
# ============================================================
# BLOQUE 7. DIAGNÓSTICO INICIAL DE CODIFICACIÓN
# ============================================================

diagnostico_valores = []

for var in variables_cultura_esperadas:
    if var in df_cc.columns:
        conteo = df_cc[var].value_counts(dropna=False).reset_index()
        conteo.columns = ["valor", "frecuencia"]
        conteo["variable"] = var
        conteo["pregunta"] = diccionario_cultura.get(var, "")
        diagnostico_valores.append(conteo)

diagnostico_valores = pd.concat(diagnostico_valores, ignore_index=True)

# Ordenar
diagnostico_valores = diagnostico_valores[["variable", "pregunta", "valor", "frecuencia"]]

display(diagnostico_valores)

ruta_diag = os.path.join(CARPETA_TABLAS, "diagnostico_valores_originales.xlsx")
diagnostico_valores.to_excel(ruta_diag, index=False)

print(f"✅ Diagnóstico de valores exportado: {ruta_diag}")

,variable,pregunta,valor,frecuencia
0,honestidad,Cumplimiento de normas: honestidad/legalidad e...,0.0,3821
1,honestidad,Cumplimiento de normas: honestidad/legalidad e...,1.0,2924
2,convivencia,Cumplimiento de normas básicas de convivencia,0.0,3842
3,convivencia,Cumplimiento de normas básicas de convivencia,1.0,2903
4,bienes,Cuidado y respeto de espacios y bienes públicos,0.0,4196
5,bienes,Cuidado y respeto de espacios y bienes públicos,1.0,2549
6,transito,Respeto a normas básicas de tránsito,0.0,4508
7,transito,Respeto a normas básicas de tránsito,1.0,2237
8,ambiental,Respeto a normas ambientales,0.0,4727
9,ambiental,Respeto a normas ambientales,1.0,2018


✅ Diagnóstico de valores exportado: resultados_cultura_ciudadana/tablas/diagnostico_valores_originales.xlsx


In [17]:
# ============================================================
# BLOQUE 8. RECODIFICACIÓN DE VARIABLES TRANSVERSALES
# ============================================================

df_cc_etq = df_cc.copy()

# Recodificación municipio
if "municipio" in df_cc_etq.columns:
    df_cc_etq["municipio_etq"] = df_cc_etq["municipio"].map({
        0: "Pitalito",
        1: "Garzón",
        2: "Neiva",
        "0": "Pitalito",
        "1": "Garzón",
        "2": "Neiva"
    })

# Recodificación sexo
if "sexo" in df_cc_etq.columns:
    df_cc_etq["sexo_etq"] = df_cc_etq["sexo"].map({
        0: "Masculino",
        1: "Femenino",
        "0": "Masculino",
        "1": "Femenino"
    })

# Recodificación nivel socioeconómico
if "nivels" in df_cc_etq.columns:
    df_cc_etq["nivels_etq"] = df_cc_etq["nivels"].map({
        0: "Bajo",
        1: "Medio",
        2: "Alto",
        "0": "Bajo",
        "1": "Medio",
        "2": "Alto"
    })

# Recodificación nivel educativo
if "nedu" in df_cc_etq.columns:
    df_cc_etq["nedu_etq"] = df_cc_etq["nedu"].map({
        1: "Ninguno",
        2: "Hasta primaria",
        3: "Hasta bachillerato / Técnico-Tecnológico",
        4: "Superior",
        5: "Pregrado",
        6: "Posgrado",
        "1": "Ninguno",
        "2": "Hasta primaria",
        "3": "Hasta bachillerato / Técnico-Tecnológico",
        "4": "Superior",
        "5": "Pregrado",
        "6": "Posgrado"
    })

# Crear rangos de edad
if "edad" in df_cc_etq.columns:
    df_cc_etq["edad_num"] = pd.to_numeric(df_cc_etq["edad"], errors="coerce")
    df_cc_etq["edad_rango"] = pd.cut(
        df_cc_etq["edad_num"],
        bins=[17, 25, 35, 45, 60, 120],
        labels=["18-25", "26-35", "36-45", "46-60", "61 o más"],
        right=True
    )

# Año numérico
if "an" in df_cc_etq.columns:
    df_cc_etq["an"] = pd.to_numeric(df_cc_etq["an"], errors="coerce")

print("✅ Variables transversales recodificadas.")

cols_ver = [c for c in ["municipio", "municipio_etq", "sexo", "sexo_etq", "nivels", "nivels_etq", "edad", "edad_rango", "nedu", "nedu_etq", "an"] if c in df_cc_etq.columns]
display(df_cc_etq[cols_ver].head())

✅ Variables transversales recodificadas.


,municipio,municipio_etq,sexo,sexo_etq,nivels,nivels_etq,edad,edad_rango,nedu,nedu_etq,an
0,1,Garzón,0,Masculino,1,Medio,31,26-35,NaN,NaN,2021
1,1,Garzón,1,Femenino,0,Bajo,21,18-25,NaN,NaN,2021
2,1,Garzón,1,Femenino,1,Medio,27,26-35,NaN,NaN,2021
3,1,Garzón,1,Femenino,1,Medio,19,18-25,NaN,NaN,2021
4,1,Garzón,1,Femenino,0,Bajo,22,18-25,NaN,NaN,2021


In [18]:
# ============================================================
# BLOQUE 9. RECODIFICACIÓN DE VARIABLES DE CULTURA CIUDADANA
# ============================================================

def convertir_binaria(valor):
    """
    Convierte valores 0/1 a etiquetas No/Sí.
    Conserva NA como NaN.
    También admite textos si vienen escritos.
    """
    if pd.isna(valor):
        return np.nan

    val = str(valor).strip().lower()
    val = unidecode(val)

    if val in ["0", "0.0", "no"]:
        return "No"
    elif val in ["1", "1.0", "si", "sí"]:
        return "Sí"
    else:
        return np.nan

# Crear columnas etiquetadas y numéricas depuradas
for var in variables_cultura_esperadas:
    if var in df_cc_etq.columns:
        df_cc_etq[f"{var}_num"] = pd.to_numeric(df_cc_etq[var], errors="coerce")
        df_cc_etq[f"{var}_etq"] = df_cc_etq[var].apply(convertir_binaria)

print("✅ Variables de cultura ciudadana recodificadas.")

# Verificación rápida
cols_etq = [f"{v}_etq" for v in variables_cultura_esperadas if f"{v}_etq" in df_cc_etq.columns]
display(df_cc_etq[cols_etq].head())

✅ Variables de cultura ciudadana recodificadas.


,honestidad_etq,convivencia_etq,bienes_etq,transito_etq,ambiental_etq,migrantes_etq,pensamiento_etq,impuestos_etq,arrojar_etq,conexion_etq,orinar_etq,indebido_etq,construccion_etq,volumen_etq,compotro_etq
0,Sí,Sí,Sí,No,Sí,NaN,NaN,Sí,Sí,Sí,Sí,Sí,Sí,Sí,NaN
1,No,No,No,No,No,NaN,NaN,No,Sí,Sí,Sí,No,No,Sí,NaN
2,No,Sí,No,No,No,NaN,NaN,Sí,Sí,Sí,Sí,Sí,Sí,Sí,NaN
3,No,No,No,No,No,NaN,NaN,Sí,Sí,Sí,Sí,Sí,Sí,No,NaN
4,Sí,Sí,No,No,Sí,NaN,NaN,No,No,No,No,No,No,No,NaN


In [19]:
# ============================================================
# BLOQUE 10. DIAGNÓSTICO DE NA Y DENOMINADORES
# ============================================================

resumen_na = []

N_total = len(df_cc_etq)

for var in variables_cultura_esperadas:
    col = f"{var}_etq"
    if col in df_cc_etq.columns:
        n_valido = df_cc_etq[col].notna().sum()
        n_na = df_cc_etq[col].isna().sum()
        pct_valido = n_valido / N_total * 100
        pct_na = n_na / N_total * 100

        resumen_na.append({
            "variable": var,
            "nombre_corto": nombres_cortos.get(var, var),
            "grupo": grupo_variable.get(var, ""),
            "pregunta": diccionario_cultura.get(var, ""),
            "n_total_base": N_total,
            "n_valido_denominador": n_valido,
            "n_na": n_na,
            "pct_valido": pct_valido,
            "pct_na": pct_na,
            "nota_denominador": "Los porcentajes de Sí/No se calculan sobre casos válidos, excluyendo NA = No aplica / Sin dato."
        })

resumen_na = pd.DataFrame(resumen_na)

display(resumen_na)

ruta_na = os.path.join(CARPETA_TABLAS, "resumen_na_denominadores.xlsx")
resumen_na.to_excel(ruta_na, index=False)

print(f"✅ Resumen de NA y denominadores exportado: {ruta_na}")

,variable,nombre_corto,grupo,pregunta,n_total_base,n_valido_denominador,n_na,pct_valido,pct_na,nota_denominador
0,honestidad,Conexión legal a servicios,Cumplimiento de normas,Cumplimiento de normas: honestidad/legalidad e...,6745,6745,0,100.000000,0.000000,Los porcentajes de Sí/No se calculan sobre cas...
1,convivencia,Convivencia,Cumplimiento de normas,Cumplimiento de normas básicas de convivencia,6745,6745,0,100.000000,0.000000,Los porcentajes de Sí/No se calculan sobre cas...
2,bienes,Bienes públicos,Cumplimiento de normas,Cuidado y respeto de espacios y bienes públicos,6745,6745,0,100.000000,0.000000,Los porcentajes de Sí/No se calculan sobre cas...
3,transito,Normas de tránsito,Cumplimiento de normas,Respeto a normas básicas de tránsito,6745,6745,0,100.000000,0.000000,Los porcentajes de Sí/No se calculan sobre cas...
4,ambiental,Normas ambientales,Cumplimiento de normas,Respeto a normas ambientales,6745,6745,0,100.000000,0.000000,Los porcentajes de Sí/No se calculan sobre cas...
5,migrantes,Migrantes/refugiados,Cumplimiento de normas,Cumplimiento de normas / comportamiento frente...,6745,5095,1650,75.537435,24.462565,Los porcentajes de Sí/No se calculan sobre cas...
6,pensamiento,Diferencia política,Cumplimiento de normas,Cumplimiento de normas / respeto frente a quie...,6745,5095,1650,75.537435,24.462565,Los porcentajes de Sí/No se calculan sobre cas...
7,impuestos,Castigo por impuestos,Percepción de sanción,Cree que puede ser castigado por no pagar impu...,6745,6745,0,100.000000,0.000000,Los porcentajes de Sí/No se calculan sobre cas...
8,arrojar,Castigo por basura,Percepción de sanción,Cree que puede ser castigado por arrojar basur...,6745,6745,0,100.000000,0.000000,Los porcentajes de Sí/No se calculan sobre cas...
9,conexion,Castigo por conexión ilegal,Percepción de sanción,Cree que puede ser castigado por conectarse il...,6745,6745,0,100.000000,0.000000,Los porcentajes de Sí/No se calculan sobre cas...


✅ Resumen de NA y denominadores exportado: resultados_cultura_ciudadana/tablas/resumen_na_denominadores.xlsx


In [20]:
# ============================================================
# BLOQUE 11. FUNCIONES DE APOYO PARA TABLAS Y GRÁFICOS
# ============================================================

PALETA_BINARIA = {
    "Sí": "#1F77B4",
    "No": "#B0B0B0"
}

PALETA_MUNICIPIO = {
    "Garzón": "#1F77B4",
    "Neiva": "#FF7F0E",
    "Pitalito": "#2CA02C"
}

def guardar_figura(nombre_archivo):
    """
    Guarda la figura actual en PNG y PDF.
    """
    ruta_png = os.path.join(CARPETA_GRAFICOS, f"{nombre_archivo}.png")
    ruta_pdf = os.path.join(CARPETA_GRAFICOS, f"{nombre_archivo}.pdf")
    plt.savefig(ruta_png, dpi=300, bbox_inches="tight")
    plt.savefig(ruta_pdf, bbox_inches="tight")
    print(f"✅ Gráfico guardado: {ruta_png}")
    print(f"✅ Gráfico guardado: {ruta_pdf}")


def tabla_frecuencia_variable(data, var):
    """
    Calcula frecuencia y porcentaje para una variable dicotómica etiquetada.
    """
    col = f"{var}_etq"
    temp = data[col].value_counts(dropna=False).reset_index()
    temp.columns = ["respuesta", "n"]
    temp["variable"] = var
    temp["nombre_corto"] = nombres_cortos.get(var, var)
    temp["grupo"] = grupo_variable.get(var, "")
    temp["pregunta"] = diccionario_cultura.get(var, "")
    temp["n_total"] = len(data)
    temp["pct_sobre_total"] = temp["n"] / len(data) * 100

    n_valido = data[col].notna().sum()
    temp["n_valido"] = n_valido
    temp["pct_sobre_validos"] = np.where(
        temp["respuesta"].notna(),
        temp["n"] / n_valido * 100 if n_valido > 0 else np.nan,
        np.nan
    )

    return temp[[
        "variable", "nombre_corto", "grupo", "pregunta",
        "respuesta", "n", "n_total", "n_valido",
        "pct_sobre_total", "pct_sobre_validos"
    ]]


def porcentaje_si_por_variable(data, variables):
    """
    Calcula porcentaje de Sí para cada variable.
    """
    filas = []

    for var in variables:
        col = f"{var}_etq"
        if col in data.columns:
            serie = data[col]
            n_valido = serie.notna().sum()
            n_si = (serie == "Sí").sum()
            n_no = (serie == "No").sum()

            pct_si = n_si / n_valido * 100 if n_valido > 0 else np.nan
            pct_no = n_no / n_valido * 100 if n_valido > 0 else np.nan

            filas.append({
                "variable": var,
                "nombre_corto": nombres_cortos.get(var, var),
                "grupo": grupo_variable.get(var, ""),
                "pregunta": diccionario_cultura.get(var, ""),
                "n_valido": n_valido,
                "n_si": n_si,
                "n_no": n_no,
                "pct_si": pct_si,
                "pct_no": pct_no
            })

    return pd.DataFrame(filas)


def cruzar_por_grupo(data, var, grupo):
    """
    Calcula porcentaje de Sí por una variable de agrupación.
    Ejemplo: variable cultura x municipio_etq.
    """
    col = f"{var}_etq"

    temp = data[[grupo, col]].dropna(subset=[grupo]).copy()

    salida = (
        temp
        .groupby(grupo, dropna=False)
        .agg(
            n_total_grupo=(col, "size"),
            n_valido=(col, lambda x: x.notna().sum()),
            n_si=(col, lambda x: (x == "Sí").sum()),
            n_no=(col, lambda x: (x == "No").sum())
        )
        .reset_index()
    )

    salida["pct_si"] = np.where(
        salida["n_valido"] > 0,
        salida["n_si"] / salida["n_valido"] * 100,
        np.nan
    )

    salida["pct_no"] = np.where(
        salida["n_valido"] > 0,
        salida["n_no"] / salida["n_valido"] * 100,
        np.nan
    )

    salida["variable"] = var
    salida["nombre_corto"] = nombres_cortos.get(var, var)
    salida["grupo_variable"] = grupo_variable.get(var, "")
    salida["pregunta"] = diccionario_cultura.get(var, "")
    salida["variable_cruce"] = grupo

    return salida


def agregar_etiquetas_barras(ax, formato="{:.1f}%"):
    """
    Agrega etiquetas encima de las barras.
    """
    for p in ax.patches:
        height = p.get_height()
        if pd.notna(height) and height > 0:
            ax.annotate(
                formato.format(height),
                (p.get_x() + p.get_width() / 2., height),
                ha="center",
                va="bottom",
                fontsize=9,
                xytext=(0, 3),
                textcoords="offset points"
            )


def fuente_pie(n, extra=""):
    """
    Pie de fuente estándar.
    """
    base = f"Fuente: Encuesta de Calidad de Vida, USCO — elaboración propia. n = {n:,}."
    if extra:
        base += f" {extra}"
    return base

In [21]:
# ============================================================
# BLOQUE 12. TABLAS DE FRECUENCIA PARA TODAS LAS VARIABLES
# ============================================================

tablas_frecuencia = []

for var in variables_cultura_esperadas:
    col = f"{var}_etq"
    if col in df_cc_etq.columns:
        tablas_frecuencia.append(tabla_frecuencia_variable(df_cc_etq, var))

tabla_frecuencias_general = pd.concat(tablas_frecuencia, ignore_index=True)

display(tabla_frecuencias_general)

ruta_freq = os.path.join(CARPETA_TABLAS, "frecuencias_todas_las_variables.xlsx")
tabla_frecuencias_general.to_excel(ruta_freq, index=False)

print(f"✅ Tabla general de frecuencias exportada: {ruta_freq}")

,variable,nombre_corto,grupo,pregunta,respuesta,n,n_total,n_valido,pct_sobre_total,pct_sobre_validos
0,honestidad,Conexión legal a servicios,Cumplimiento de normas,Cumplimiento de normas: honestidad/legalidad e...,No,3821,6745,6745,56.649370,56.649370
1,honestidad,Conexión legal a servicios,Cumplimiento de normas,Cumplimiento de normas: honestidad/legalidad e...,Sí,2924,6745,6745,43.350630,43.350630
2,convivencia,Convivencia,Cumplimiento de normas,Cumplimiento de normas básicas de convivencia,No,3842,6745,6745,56.960712,56.960712
3,convivencia,Convivencia,Cumplimiento de normas,Cumplimiento de normas básicas de convivencia,Sí,2903,6745,6745,43.039288,43.039288
4,bienes,Bienes públicos,Cumplimiento de normas,Cuidado y respeto de espacios y bienes públicos,No,4196,6745,6745,62.209044,62.209044
5,bienes,Bienes públicos,Cumplimiento de normas,Cuidado y respeto de espacios y bienes públicos,Sí,2549,6745,6745,37.790956,37.790956
6,transito,Normas de tránsito,Cumplimiento de normas,Respeto a normas básicas de tránsito,No,4508,6745,6745,66.834692,66.834692
7,transito,Normas de tránsito,Cumplimiento de normas,Respeto a normas básicas de tránsito,Sí,2237,6745,6745,33.165308,33.165308
8,ambiental,Normas ambientales,Cumplimiento de normas,Respeto a normas ambientales,No,4727,6745,6745,70.081542,70.081542
9,ambiental,Normas ambientales,Cumplimiento de normas,Respeto a normas ambientales,Sí,2018,6745,6745,29.918458,29.918458


✅ Tabla general de frecuencias exportada: resultados_cultura_ciudadana/tablas/frecuencias_todas_las_variables.xlsx
